# Implementing LSH on the N-most cited dataset

In [1]:
import json
from typing import Dict, Any
import numpy as np

In [2]:
from lsh import preprocess_lsh, lsh 
from lsh_utils.signatures import signatures
from lsh_utils.Jaccard_similarity import Jaccard_similarity_shingles

## Preprocess 

Keep ony the important part of the data

In [4]:
json_path_Nmost = '../data/processed/filtered_articles_Nmostcited.json'
data_Nmost = preprocess_lsh(dataset_path = json_path_Nmost)
print("preprocessing done")

Data succesfully loaded
preprocessing done


Save the preprocessed data into a json file

In [5]:
output_json_path = '../data/processed/sub_datasets_lsh/data_Nmost.json'
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_Nmost, f, indent=4)

## Compute the signatures

Load the simplified data previously saved

In [3]:
data_Nmost_path = '../data/processed/sub_datasets_lsh/data_Nmost.json'
with open(data_Nmost_path, 'r', encoding='utf-8') as f:
            data_Nmost: Dict[str, Any] = json.load(f)

Compute the signature matrix

In [21]:
q = 7 
b = 10
r = 10

signature_matrix_Nmost, idx_to_id_Nmost = signatures(
    doc_list=data_Nmost,
    shingle_size = q,
    signature_size = b*r
    )

Computing signatures: 100%|██████████| 24290/24290 [57:26<00:00,  7.05it/s]   

min hashing of the documents complete


Save the results

In [22]:
np.save(file = f"../data/processed/signatures_lsh/Nmost_q{q}_b{b}_r{r}", arr = signature_matrix_Nmost)
output_json_path = f"../data/processed/signatures_lsh/idx_to_id_Nmost_q{q}_b{b}_r{r}.json"
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_Nmost, f, indent=4)

## Perform the research of the most relevant documents using LSH

### Try with the parmeters : q = 7, b = 4, r = 5

load data and the saved signature matrix

In [6]:
data_Nmost_path = '../data/processed/sub_datasets_lsh/data_Nmost.json'
with open(data_Nmost_path, 'r', encoding='utf-8') as f:
            data_Nmost: Dict[str, Any] = json.load(f)

q = 7 
b = 4
r = 5

try :
    signature_matrix_Nmost = np.load(f"../data/processed/signatures_lsh/Nmost_q{q}_b{b}_r{r}.npy")
    with open(f"../data/processed/signatures_lsh/idx_to_id_Nmost_q{q}_b{b}_r{r}.json", 'r', encoding='utf-8') as f:
            idx_to_id_Nmost: Dict[str, Any] = json.load(f)
except FileNotFoundError as e :
    print(f"No signature matrix has been saved with the set of parameters : q = {q}, b = {b}, r = {r} ")

In [7]:
print(signature_matrix_Nmost.shape)

(20, 24290)


In [8]:
print(idx_to_id_Nmost[2])

{'id': '0911.0802', 'abstract': 'we construct analytic extensions of the pomeranskysenkov metrics with\nmultiple killing horizons and asymptotic regions we show that in our\nextensions the singularities associated to an obstruction to differentiability\nof the metric lie beyond event horizons we analyze the topology of the\nnonempty singular set which turns out to be parameterdependent we present\nnumerical evidence for stable causality of the domain of outer communications\nthe resulting global structure is somewhat reminiscent of that of kerr\nspacetime'}


Perform LSH to obtain the most similar documents to input (try with abstract of the first document as input)

In [9]:
Most_similar_20, Scores = lsh(
        input = data_Nmost[0]['abstract'],
        signature_matrix = signature_matrix_Nmost,
        idx_to_id = idx_to_id_Nmost,
        m = signature_matrix_Nmost.shape[1]//10, 
        shingle_size = q,
        nb_band = b,
        band_size = r,
        )

Computing the signature of every document in the dataset ...
Computing signature of input ...


Computing signatures: 100%|██████████| 1/1 [00:00<00:00, 64.24it/s]


min hashing of the documents complete
Performing LSH to find similar candidates ...


LSH Bands:  25%|██▌       | 1/4 [00:00<00:00,  6.79it/s]

4.1169205434335116e-05 % of signatures computed ... 



...

LSH Bands:  75%|███████▌  | 3/4 [00:00<00:00,  6.72it/s]

2.0000411692054345 % of signatures computed ... 



LSH Bands: 100%|██████████| 4/4 [00:00<00:00,  6.74it/s]


3.0000411692054345 % of signatures computed ... 

LSH successfully performed to find similar candidates
Calculation of the actual similarities ...


Calculating Similarities: 100%|██████████| 45/45 [00:00<00:00, 103933.74it/s]


In [10]:
print("Most similar documents : " , Most_similar_20, '\n')
print("Scores : " , Scores)

Most similar documents :  ['1002.3982', '0802.1718', '1006.0106', '0806.3825', '1010.5141', '1110.4738', '1203.1672', '0807.0646', '0808.1161', '0904.0380', '0808.3413', '0811.4571', '1110.6838', '1103.0885', '1111.5608', '0802.4221', '0904.3198', '1005.3266', '0901.4348', '0903.3108', '0906.1523', '0903.2733', '0907.5115', '0706.0322', '0810.3296', '1005.4655', '0906.4370', '1003.3155', '0712.2716', '0903.4375', '1003.3590', '1202.1316', '0704.3084', '1001.3651', '1012.1201', '1108.2291', '0909.1739', '0806.4688', '1011.5154', '0803.1447', '0805.0332', '0807.4146', '1005.1132', '0902.1539', '0809.5268'] 

Scores :  [1.0, 0.15, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [11]:
data_Nmost[0]['id'] # id of the first document (input)

'1002.3982'

We notice that the document has a 100% similarity with himself ! Let's have a look to the second most relevant document found

In [12]:
def find_index(L, x):
    """
    Calculates the index (i) of the first occurrence of element x in list L.

    Args:
        L (list): The list to search within.
        x (any): The element whose index is being sought.

    Returns:
        int: The index of element x in L.

    Raises:
        ValueError: If element x is not found in list L.
    """
    try:
        # The index() method returns the index of the first occurrence
        # of the specified element.
        index = L.index(x)
        return index
    except ValueError:
        # index() raises a ValueError if the element is not found.
        # It's good practice to handle this error.
        raise ValueError(f"The element '{x}' is not in the list.")

In [13]:
Id_list = [doc["id"] for doc in idx_to_id_Nmost]
i = find_index(Id_list,Most_similar_20[1])
abstract_20 = data_Nmost[i]["abstract"]
print(abstract_20)

we present a systematic group space scan of discrete abelian flavor
symmetries for lepton mass models that produce nearly tribimaximal lepton
mixing in our models small neutrino masses are generated by the typei seesaw
mechanism the lepton mass matrices emerge from higherdimension operators via
the froggattnielsen mechanism and are predicted as powers of a single
expansion parameter epsilon that is of the order of the cabibbo angle
thetacsimeq 02 we focus on solutions that can give close to tribimaximal
lepton mixing with a very small reactor angle theta13approx 0 and find
several thousand explicit such models that provide an excellent fit to current
neutrino data the models are rather general in the sense that large leptonic
mixings can come from the charged leptons andor neutrinos moreover in the
neutrino sector both left and righthanded neutrinos can mix maximally we
also find a new relation theta13lesssimepsilon3 for the reactor angle
and a new sum rule theta23approxpi4epsilonsqrt2

This document is about neutrinos

In [14]:
print(data_Nmost[0]["abstract"])

new particles at the tev scale can decay hadronically with strongly
collimated jets thus the standard reconstruction methods based on
invariantmasses of wellseparated jets can fail we discuss how to identify
such particles in pp collisions at the lhc using jet shapes which help to
reduce the contribution of qcdinduced events we focus on a rather generic
example x to ttbar to hadrons with x being a heavy particle but the approach
is well suited for reconstruction of other decay channels characterized by a
cascade decay of known states


The original document talks about particles : which is a similar subject !!

### Try with the set of parameters : q = 7, b = 10, r = 10

In [15]:
data_Nmost_path = '../data/processed/sub_datasets_lsh/data_Nmost.json'
with open(data_Nmost_path, 'r', encoding='utf-8') as f:
            data_Nmost: Dict[str, Any] = json.load(f)

q = 7 
b = 10
r = 10

try :
    signature_matrix_Nmost = np.load(f"../data/processed/signatures_lsh/Nmost_q{q}_b{b}_r{r}.npy")
    with open(f"../data/processed/signatures_lsh/idx_to_id_Nmost_q{q}_b{b}_r{r}.json", 'r', encoding='utf-8') as f:
            idx_to_id_Nmost: Dict[str, Any] = json.load(f)
except FileNotFoundError as e :
    print(f"No signature matrix has been saved with the set of parameters : q = {q}, b = {b}, r = {r} ")

In [16]:
print(signature_matrix_Nmost.shape)

(100, 24290)


Perform LSH to obtain the most similar documents to input (try with abstract of the first document as input)

In [17]:
Most_similar_100, Scores = lsh(
        input = data_Nmost[0]['abstract'],
        signature_matrix = signature_matrix_Nmost,
        idx_to_id = idx_to_id_Nmost,
        m = signature_matrix_Nmost.shape[1]//10, 
        shingle_size = q,
        nb_band = b,
        band_size = r,
        )

Computing the signature of every document in the dataset ...
Computing signature of input ...


Computing signatures: 100%|██████████| 1/1 [00:00<00:00, 14.27it/s]


min hashing of the documents complete
Performing LSH to find similar candidates ...


LSH Bands:  10%|█         | 1/10 [00:00<00:01,  4.74it/s]

4.1169205434335116e-05 % of signatures computed ... 



...

LSH Bands:  90%|█████████ | 9/10 [00:01<00:00,  4.56it/s]

8.000041169205435 % of signatures computed ... 



LSH Bands: 100%|██████████| 10/10 [00:02<00:00,  4.51it/s]


9.000041169205435 % of signatures computed ... 

LSH successfully performed to find similar candidates
Calculation of the actual similarities ...


Calculating Similarities: 100%|██████████| 107/107 [00:00<00:00, 30963.88it/s]


In [18]:
print("Most similar documents : " , Most_similar_100, '\n')
print("Scores : " , Scores)

Most similar documents :  ['1002.3982', '1001.4577', '1201.3339', '1112.3024', '0908.0880', '0710.2435', '0711.1365', '0806.0050', '0706.4153', '1107.1244', '0812.3886', '1207.4235', '0807.3834', '0901.4533', '0710.0915', '0903.1286', '1008.1632', '0910.0554', '1106.5493', '0709.0007', '0907.2997', '1001.3651', '0803.4323', '0905.0059', '1107.1997', '1012.4840', '1110.6249', '1003.4660', '0711.4754', '0911.0352', '0712.4328', '0811.3665', '0909.4766', '0805.2466', '0909.2937', '0705.2629', '0708.4003', '1112.3351', '0806.4175', '0911.1535', '1004.0162', '0801.0778', '0906.2165', '0804.2519', '1111.5698', '0705.0572', '1004.2045', '0705.3075', '0712.1026', '0903.3727', '1109.0862', '0912.0399', '0709.2307', '0912.1608', '0905.0188', '0712.1028', '0907.1014', '1010.0055', '0912.5054', '0805.4451', '0904.1554', '0907.5008', '1102.0342', '0807.1939', '0910.2076', '0807.1108', '0709.0731', '0907.1269', '1108.6269', '0810.4846', '0711.2369', '1207.1468', '0909.1063', '0903.0801', '1002.2424'

In [19]:
data_Nmost[0]['id'] # id of the first document (input)

'1002.3982'

In [20]:
def find_index(L, x):
    """
    Calculates the index (i) of the first occurrence of element x in list L.

    Args:
        L (list): The list to search within.
        x (any): The element whose index is being sought.

    Returns:
        int: The index of element x in L.

    Raises:
        ValueError: If element x is not found in list L.
    """
    try:
        # The index() method returns the index of the first occurrence
        # of the specified element.
        index = L.index(x)
        return index
    except ValueError:
        # index() raises a ValueError if the element is not found.
        # It's good practice to handle this error.
        raise ValueError(f"The element '{x}' is not in the list.")
Id_list = [doc["id"] for doc in idx_to_id_Nmost]
i = find_index(Id_list,Most_similar_100[1])
abstract_100 = data_Nmost[i]["abstract"]
print(abstract_100)

we report a search for single top quark production with the cdf ii detector
using 21 fb1 of integrated luminosity of pbar p collisions at sqrts196
tev the data selected consist of events characterized by large energy
imbalance in the transverse plane and hadronic jets and no identified
electrons and muons so the sample is enriched in w  tau nu decays in order
to suppress backgrounds additional kinematic and topological requirements are
imposed through a neural network and at least one of the jets must be
identified as a bquark jet we measure an excess of signallike events in
agreement with the standard model prediction but inconsistent with a model
without single top quark production by 21 standard deviations sigma with a
median expected sensitivity of 14 sigma assuming a top quark mass of 175
gevc2 and ascribing the excess to single top quark production the cross
section is measured to be 492522statsystpb consistent with
measurements performed in independent datasets and with the stan

In [21]:
print(data_Nmost[0]["abstract"])

new particles at the tev scale can decay hadronically with strongly
collimated jets thus the standard reconstruction methods based on
invariantmasses of wellseparated jets can fail we discuss how to identify
such particles in pp collisions at the lhc using jet shapes which help to
reduce the contribution of qcdinduced events we focus on a rather generic
example x to ttbar to hadrons with x being a heavy particle but the approach
is well suited for reconstruction of other decay channels characterized by a
cascade decay of known states


We also find a similar subject for the most relevant document

### Comparing the results of b,r = 4,5 and b,r = 10,10 for the second most similar document

In [22]:
from lsh_utils.shingle import shingle

In [23]:
input_content = data_Nmost[0]["abstract"]
shingle_input = shingle(q=7, text = input_content)
shingle_20 = shingle(q=7, text = abstract_20)
shingle_100 = shingle(q=7,text = abstract_100)

In [24]:
Sim_20 = Jaccard_similarity_shingles(shingles_list_A=shingle_input,shingles_list_B=shingle_20)
Sim_100 = Jaccard_similarity_shingles(shingles_list_A=shingle_input, shingles_list_B=shingle_100)

In [25]:
print(Sim_20,Sim_100)

0.006074411541381929 0.021702838063439065


The set of parameters with b,r = 10,10 has far better results !